# Libraries

In [1]:
# 0. Clean slate — remove existing copies before reinstalling
!pip uninstall -y torch torchvision torchaudio torchao

!pip install -q torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 torchao==0.16.0 --index-url https://download.pytorch.org/whl/cu128

# 3. Evaluation Framework
!pip install -q "lm_eval[hf]"

!pip install -q "cuda-core==0.3.*" "numba-cuda[cu12]<0.23.0,>=0.22.2" "numba<0.62.0,>=0.60.0"

!pip install -q transformers==4.57.6

!pip install -q langdetect

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 916.9/916.9 MB 1.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 5.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 1.6 MB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 7.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 108.1 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver 

In [2]:
# Libraries
import gc
import glob
import shutil
import os
import random
import subprocess

import numpy as np
import torch
import lm_eval

from dataclasses import dataclass
from typing import Optional
from huggingface_hub import login
from IPython.display import FileLink
from kaggle_secrets import UserSecretsClient

# Global Variables

In [ ]:
# --- Identity ---
user_name = "Abdelrahmanemam01"
email     = "abdulrahmanemam01@gmail.com"
repo      = "MahmoudOsama20/EdgeCompress"  # Fixed

# --- Model ---
model      = "Phi-4-mini-instruct"
MODEL_NAME       = "Phi-4-mini-instruct-Original"
MODEL_PRETRAINED = "microsoft/Phi-4-mini-instruct"
TOKENIZER_ID = "microsoft/Phi-4-mini-instruct"
MODEL_PRETRAINED_FIX = MODEL_PRETRAINED.replace("/", "__")

# --- Reproducibility ---
SEED = 42

# --- Environment ---
os.environ["HF_ALLOW_CODE_EVAL"] = "1"

# Classes

In [4]:
@dataclass
class Task:
    key:               str
    name:              str
    category:          str
    num_fewshot:       int  = 0
    batch_size:        int  = 2
    limit:             Optional[int] = None
    apply_chat_template: bool = True
    fewshot_as_multiturn: bool = False
    max_gen_toks:      Optional[int] = None    

# Functions

In [5]:
# REPRODUCIBILITY
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to {seed}")

# Seeding

In [6]:
set_reproducibility(SEED)

Global seed set to 42


# Huggingface login

In [7]:
# HUGGING FACE AUTHENTICATION
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logging into Hugging Face...")

Logging into Hugging Face...


# GitHub login

In [8]:
token = UserSecretsClient().get_secret("GIT_TOKEN")
print("Logging into GitHub...")

Logging into GitHub...


# Evaluations

**Configurations**

In [ ]:
TASKS = [
    Task("arc_challenge_chat",     "arc_challenge_chat",     "reasoning",   num_fewshot=0, batch_size=16, fewshot_as_multiturn=False, max_gen_toks=1024),
]

# Model args
MODEL_ARGS = ",".join([
    f"pretrained={MODEL_PRETRAINED}",
    f"tokenizer={TOKENIZER_ID}",
    "dtype=float16",
    "parallelize=True",
    "max_length=4096",
])

results_dir = f"evaluation_results/{MODEL_NAME}"

**Evaluation loop**

In [10]:
for t in TASKS:
    print(f"\n[{t.category.upper()}] {t.key}")

    cmd = [
        "lm_eval",
        "--model",       "hf",
        "--model_args",  MODEL_ARGS,
        "--tasks",       t.name,
        "--num_fewshot", str(t.num_fewshot),
        "--batch_size",  str(t.batch_size),
        "--seed",        str(SEED),
        "--verbosity", "DEBUG",
        "--output_path", f"{results_dir}/{t.key}",
    ]

    if t.limit is not None:
        cmd += ["--limit", str(t.limit)]
        
    if t.apply_chat_template:
            cmd.append("--apply_chat_template")
        
    if t.fewshot_as_multiturn:
            cmd.append("--fewshot_as_multiturn")
        
    if t.max_gen_toks is not None:                             
        cmd += ["--gen_kwargs", f"max_gen_toks={t.max_gen_toks}"]

    subprocess.run(cmd, check=True)
    
    # Push To GitHub
    target_dir_in_repo = f"Evaluations/BenchmarkEvaluations/FinalEvaluation/{model}/{MODEL_NAME}/results/{t.key}"
    source_folder = f"{results_dir}/{t.key}/{MODEL_PRETRAINED_FIX}"  # The Kaggle folder you want to upload
    remote_url = f"https://{token}@github.com/{repo}.git"
    
    # A temporary workspace in Kaggle to handle the cloning
    temp_repo_dir = "/kaggle/working/temp_git_repo" 
    # ---------------------

    # 2. Clean up any previous runs and clone the existing repository
    os.system(f"rm -rf {temp_repo_dir}")
    os.system(f"git clone {remote_url} {temp_repo_dir}")
    
    # 3. Create the target directory inside the cloned repo
    full_target_path = f"{temp_repo_dir}/{target_dir_in_repo}"
    os.makedirs(full_target_path, exist_ok=True)
    
    # 4. Copy the contents of your Kaggle folder into the target GitHub folder
    # (Using cp -r to recursively copy all files and subfolders)
    os.system(f"cp -r {source_folder}/* {full_target_path}/")
    
    # 5. Configure, commit, and push the changes
    os.system(f"""
        git -C {temp_repo_dir} config user.email "{email}" && \
        git -C {temp_repo_dir} config user.name "{user_name}" && \
        git -C {temp_repo_dir} add . && \
        git -C {temp_repo_dir} commit -m "Add Kaggle results to {target_dir_in_repo}" && \
        git -C {temp_repo_dir} push origin main
    """)
    
    print(f"Successfully pushed to {repo} inside the '{target_dir_in_repo}' folder!")
    
    torch.cuda.empty_cache()
    gc.collect()
    print("Done")


[REASONING] arc_challenge_chat


2026-06-07:15:24:40 INFO     [config.evaluate_config:307] Using default fewshot_as_multiturn=True.
2026-06-07:15:24:51 INFO     [_cli.run:388] Selected Tasks: ['arc_challenge_chat']
2026-06-07:15:24:51 INFO     [evaluator:162] Setting verbosity through simple_evaluate is deprecated.
2026-06-07:15:24:53 INFO     [evaluator:214] Setting random seed to 42 | Setting numpy seed to 42 | Setting torch manual seed to 42 | Setting fewshot manual seed to 42
2026-06-07:15:24:53 WARNING  [evaluator:226] generation_kwargs: {'max_gen_toks': 1024} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
2026-06-07:15:24:53 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': 'Qwen/Qwen3-4B', 'tokenizer': 'Qwen/Qwen3-4B', 'dtype': 'bfloat16', 'parallelize': True, 'max_length': 4096, 'enable_thinking': False}
2026-06-07:15:24:58 INFO     [models.huggingface:306] Using `accelerate launch` or `parallelize=Tr

hf ({'pretrained': 'Qwen/Qwen3-4B', 'tokenizer': 'Qwen/Qwen3-4B', 'dtype': 'bfloat16', 'parallelize': True, 'max_length': 4096, 'enable_thinking': False}), gen_kwargs: ({'max_gen_toks': 1024}), limit: None, num_fewshot: 0, batch_size: 16
|      Tasks       |Version|     Filter      |n-shot|  Metric   |   |Value|   |Stderr|
|------------------|------:|-----------------|-----:|-----------|---|----:|---|-----:|
|arc_challenge_chat|      1|remove_whitespace|     0|exact_match|↑  |0.878|±  |0.0096|



Cloning into '/kaggle/working/temp_git_repo'...


[main 384ce90] Add Kaggle results to Evaluations/BenchmarkEvaluations/FinalEvaluation/Qwen3-4B/Qwen3-4B-Original/results/arc_challenge_chat
 1 file changed, 161 insertions(+)
 create mode 100644 Evaluations/BenchmarkEvaluations/FinalEvaluation/Qwen3-4B/Qwen3-4B-Original/results/arc_challenge_chat/results_2026-06-07T15-36-11.475073.json
Successfully pushed to MahmoudOsama20/EdgeCompress inside the 'Evaluations/BenchmarkEvaluations/FinalEvaluation/Qwen3-4B/Qwen3-4B-Original/results/arc_challenge_chat' folder!
Done


To https://github.com/MahmoudOsama20/EdgeCompress.git
   562be53..384ce90  main -> main


# Save reports

In [11]:
zip_path = shutil.make_archive(
    "evaluation_results",   # zip file name
    "zip",                  # format
    "evaluation_results"    # folder to compress
)